# V7 lightweight results: no saved model weights
Same frozen V7 experiment: reuse10 verified jobs and regenerate up to38 missing jobs. Save predictions, scores, hyperparameters and small audit/history records. NO model weights or datasets are copied. Expected output a few MB. Original results remain read-only. Use a GPU runtime and mount the account with free space.


## 1. Setup (no training)
Reuses existing cloud checkout/dependencies when possible. Never runs training on your Mac. No automatic library upgrade or version pin override. A runtime fingerprint mismatch is a stop, not permission to bypass it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json
repo = Path('/content/teco-v7-light-recovery')
tag = 'v7-v8-light-results-v1'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', tag, 'https://github.com/parmisbathayan/reproduce-teco-results.git', str(repo)], check=True)
assert subprocess.check_output(['git','-C',str(repo),'describe','--tags','--exact-match'], text=True).strip() == tag
assert not subprocess.check_output(['git','-C',str(repo),'status','--porcelain'],text=True).strip(), 'Checkout changed; preserve it and investigate.'
os.chdir(repo)
# Colab normally supplies these; do not upgrade a runtime matching the old run.
import numpy, pandas, sklearn, tensorflow
print({name: module.__version__ for name,module in [('numpy',numpy),('pandas',pandas),('scikit_learn',sklearn),('tensorflow',tensorflow)]})
root = Path('/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly')
local = Path('/content/teco-v7-light-results')
mirror = root/'v7_light_recovery_45_62'
command = [sys.executable, '-u', '-B', 'src/v7_recover_light.py', '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total', '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv', '--original-dir', str(root/'v7_unseen_sentences_45_62'), '--reference-dir', str(root/'v4_low_lr_confirmation_45_62'), '--local-dir', str(local), '--mirror-dir', str(mirror)]
def stream_run(cmd):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if code: raise subprocess.CalledProcessError(code, cmd)
mirror.mkdir(parents=True, exist_ok=True)
from uuid import uuid4
probe = mirror / (str(uuid4()) + "_storage_probe.txt")
probe.write_text("Light V7 storage check: confirm this text on the Drive website.")
print("CHECK ON DRIVE WEBSITE:", probe.name, probe.stat().st_size, "bytes")
print("If Thesis is not under My Drive, add its shared-folder shortcut or edit the paths before continuing.")


## 2. Audit, import and prepare — NO training
Expected on first use:10 verified jobs and38 needing regeneration. Complete-but-corrupt existing jobs stop instead of being silently rerun. Source folders remain untouched.

In [ ]:
stream_run(command + ['--prepare-only'])


## 3. Storage gate and missing-job regeneration
In the mounted account, confirm the probe created during setup has nonzero size and correct text on the Drive website. Then set the flag True below. No model weights are saved. Approximately45–90 GPU minutes for38 fits; progress prints at each job and every30seconds. No parameter search.


In [ ]:
DRIVE_PERSISTENCE_CHECKED = False
assert DRIVE_PERSISTENCE_CHECKED, 'Confirm the new storage probe on the Drive website first.'
stream_run(command + ['--storage-checked'])


## 4. Verify, compare, and prepare backup — NO training
Verify all48 archives; report old/new discrepancies without selecting a favorable run. Keep local backup until Drive WEBSITE contains all48 jobs plus metadata. The final ZIP can be downloaded manually from Colab's Files panel; no automatic Mac download.

In [ ]:
stream_run(command + ['--verify-only'])
import pandas as pd, shutil
from IPython.display import display
display(pd.read_csv(local/'summary_overall.csv'))
comparison = pd.read_csv(local/'comparison_to_original.csv')
display(comparison[~comparison.exact_confusion_match])
print('Different confusion matrices:', int((~comparison.exact_confusion_match).sum()), '/96; preserve and review these differences.')
backup = shutil.make_archive('/content/v7_light_results_complete', 'zip', local)
print('CLOUD BACKUP:', backup, 'bytes:', Path(backup).stat().st_size)
print('No training in this cell. Confirm the Drive WEBSITE has all48 job ZIPs before disconnecting.')
print('Actual local result bytes:', sum(p.stat().st_size for p in local.rglob('*') if p.is_file()))
print('No model weights saved; hyperparameters are embedded in fold_metrics.csv.')
